In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import bidsio
import sys
import copy
import pickle
sys.path.append('../')
from helpers import *
from fns_grid import *
from bandlimited_signal import *


device = 'cuda:6' if torch.cuda.is_available() else 'cpu'
seed, RES = 0, 64


In [ ]:
def load_gt(dataset, id_val, bandlimit = 0.6):
    RES = 64
    if dataset=='dragon':
        signal =  resize(Voxel_Fitting(dimension=3, length=RES, bandlimit=bandlimit, seed=id_val, super_resolution=False, sparse=True).signal, (RES, RES, RES))
    elif dataset == 'sphere':
        signal = SparseSphereSignal(dimension=3, length=RES, bandlimit=bandlimit, seed=id_val, generate=False).signal
    elif dataset == 'bandlimited':
        signal = BandlimitedSignal(dimension=3, length=RES, bandlimit=bandlimit, seed=id_val, generate=False).signal  
    return signal

dataset='dragon'
id_val = 1234
signal = load_gt(dataset, id_val)
print(signal.shape)
plt.imshow(signal[:,:,32], cmap="gray")
plt.colorbar()
plt.show()

In [ ]:
learning_rate, iters = 5e-2, 1000
rng = np.random.default_rng(seed)
slice_idx = 32

model_sizes = [52000, 105000]
outputs = {}
to_save_outputs = {}
for model_size in model_sizes:
    r = compute_params_from_model_size("grid_eta", None, model_size, n_dims=3)
    print(f'grid_reso: {r}')
    torch.manual_seed(seed)
    output = fit_grid(target_signal=signal, r = r, iters = iters, lr=learning_rate, interp="bilinear", log_interval=250, seed=0, count_params=True)
    outputs[f'{r}'] = output     
    to_save_outputs[f'{r}'] = output['best_pred'].reshape((RES, RES, RES))[:,:,slice_idx]
    error = np.linalg.norm(signal.flatten() - output['best_pred'].flatten())                       
    print(f"Error: {error:.3e}, Loss: {output['best_loss']:.3e}")

with open(f"3d_dragon/grid.pkl", "wb") as f:
    pickle.dump(to_save_outputs, f)

In [ ]:
from helpers import *

slice_idx = 32 
def slice_outputs(outputs, slice_idx):
    new_outputs = copy.deepcopy(outputs)
    small_param, large_param = list(outputs.keys())
    small_pred, large_pred   = new_outputs[small_param]['best_pred'].reshape((RES, RES, RES)), new_outputs[large_param]['best_pred'].reshape((RES, RES, RES))
    new_outputs[small_param], new_outputs[large_param] = new_outputs[small_param], new_outputs[large_param]
    new_outputs[small_param]['best_pred'] = small_pred[:,:,slice_idx]
    new_outputs[large_param]['best_pred'] = large_pred[:,:,slice_idx]
    return new_outputs

plot_error_heatmaps(signal[:,:,slice_idx], slice_outputs(outputs, slice_idx), model_name="Grid")